# Costruzione decrizione automatica svg automa

## Esempio estrazione

In [57]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo

nome_file_svg = "automa.svg"

root_svg = apri_file_svg(nome_file_svg)

for child in root_svg:
    if child.attrib['id'] == 'layer1':
        automa1 = child

#foreach element in automa1 print id
for child in automa1:
    if not re.search('title', child.attrib['id']):
        print('RAMO: ', child.attrib['id'])

        #forreach element in child print id
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                print('- ', child2.attrib['id'])
                
                #get text by id valore-r6
                for child3 in child2:
                    if re.search('valore', child3.attrib['id']):
                        for child4 in child3:
                            print('---- ', child4.text)


    print('===============================')

RAMO:  stato-q4
-  simbolo-q4
-  nome-q4
RAMO:  stato-q3
-  simbolo-q3
-  nome-q3
RAMO:  stato-q2
-  simbolo-q2
-  nome-q2
RAMO:  stato-q1
-  simbolo-q1
-  nome-q1
RAMO:  stato-q0-finale
-  simbolo-q0
-  simbolo-q0-0
-  nome-q0
RAMO:  transizione-q4-q0
-  simbolo-q4-q0
-  valore-q4-q0
RAMO:  transizione-q3-q4
-  simbolo-q3-q4
-  valore-q3-q4
RAMO:  transizione-q2-q3
-  simbolo-q2-q3
-  valore-q2-q3
RAMO:  transizione-q1-q2
-  simbolo-q1-q2
-  valore-q1-q2
RAMO:  transizione-q0-q1
-  simbolo-q0-q1
-  valore-q0-q1
RAMO:  start-q0
-  simbolo-start
-  nome-start


## GENERAZIONE REGOLE

## ESTRAZIONE DEL DIZIONARIO

In [58]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = "automa.svg"
root_svg = apri_file_svg(nome_file_svg)

# Trova l'elemento con id="automa1"
for child in root_svg:
    if child.attrib['id'] == 'layer1':
        automa1 = child


stati = []
linguaggio = []
transizioni = []
transizioni_linguaggio = []
stato_iniziale = ''
stato_finale = ''

for child in automa1:
    if not re.search('title', child.attrib['id']):
        elemento = child.attrib['id']

        if re.search('start', elemento):
            elemento = elemento.replace('start-', '')
            stato_iniziale = elemento

        if re.search('stato', elemento):
            elemento = elemento.replace('stato-', '')

            if re.search('finale', elemento):
                elemento = elemento.replace('-finale', '')
                stato_finale = elemento

            stati.append(elemento)

        if re.search('transizione', elemento):
            elemento = elemento.replace('transizione-', '')
            elemento = elemento.split('-')
            transizioni.append(elemento)

            for child2 in child:
                if re.search('valore', child2.attrib['id']):
                    for child3 in child2:
                        linguaggio.append(child3.text)

                        transizioni_linguaggio.append([elemento, child3.text])

# revert stati
stati = stati[::-1]
# revert linguaggio
linguaggio = linguaggio[::-1]
# revert transizioni
transizioni = transizioni[::-1]
# revert transizioni_linguaggio
transizioni_linguaggio = transizioni_linguaggio[::-1]


print('stato_iniziale: ', stato_iniziale)
print('stato_finale: ', stato_finale)
print('stati: ', stati)
print('linguaggio: ', linguaggio)
print('transizioni: ', transizioni)
print('transizioni_linguaggio: ', transizioni_linguaggio)

stato_iniziale:  q0
stato_finale:  q0
stati:  ['q0', 'q1', 'q2', 'q3', 'q4']
linguaggio:  ['1', '1', '0', '0', '0']
transizioni:  [['q0', 'q1'], ['q1', 'q2'], ['q2', 'q3'], ['q3', 'q4'], ['q4', 'q0']]
transizioni_linguaggio:  [[['q0', 'q1'], '1'], [['q1', 'q2'], '1'], [['q2', 'q3'], '0'], [['q3', 'q4'], '0'], [['q4', 'q0'], '0']]


In [59]:
svg = '<image>automi/automa.svg</image>'
intent = 'fsa-practical'
risposte = {}

In [60]:
categories = []

In [61]:
import random
import string

def add_typo(s):
    res = []

    # 2 versioni con un carattere rimosso
    for _ in range(2):
        idx = random.randrange(len(s))
        new = s[:idx] + s[idx+1:]
        res.append(new)

    # 2 versioni con un carattere aggiunto
    for _ in range(2):
        idx = random.randrange(len(s) + 1)
        char = random.choice(string.ascii_lowercase)  
        new = s[:idx] + char + s[idx:]
        res.append(new)

    return res


def missing_detail_trans(base, transizioni, transizioni_linguaggio):
    missing = []
    text = ' The transitions are: '
    for i in range(len(transizioni)):
        if (i == len(transizioni) - 1):
            text += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
        else:
            text += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + ', '
        missing.append(base + text)
    return missing


def missing_detail_number_trans(base, transizioni, transizioni_linguaggio):
    missing = []

    for i in range(len(transizioni)):
        text = 'The automaton has ' + str(i + 1) + ' transitions: '
        for y in range(i + 1):

            if (y == i):
                text += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
            else:
                text += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + ', '
        missing.append(base + text)
    return missing


def missing_detail_states(base, stati):
    missing = []
    for i in range(len(stati)):
        text = 'The automaton has ' + str(i+1) + ' states: '

        for y in range(i + 1):
            if (y == i):
                text += stati[y] + '.'
            elif (y == i - 1):
                text += stati[y] + ' and '
            else:
                text += stati[y] + ', '

        missing.append(base + text)
    return missing



### In generale

```xml
<category intent="fsa-practical" argument="automaton">
    <acts>
        <act time="0">Ta:request</act>
    </acts>
    <frame></frame>
    <template>
        L'automa accetta zero o più parole formate da una sequenza del tipo 11000. In totale ci sono 5 stati: q0, q1, q2, q3 e q4. q0 è 
        sia lo stato iniziale che lo stato finale.Le transizioni sono: q0 con valore 1 va in q1, q1 con valore 1 va in q2, q2 con valore
        0 va in q3, q3 con valore 0 va in q4, q4 con valore 0 va in q0.
        <image>automi/automa.svg</image>
    </template>
</category>

In [62]:
category = {
    "intent": "fsa-practical",
    "argument": "automaton",
    "acts": {
        0: "Ta:request",
    },
    "frame": "<frame></frame>",
    "template": ""
    #"request": ""
}

wrong = []
text = s1 = s2 = c = e1 = e2 = a = 'The automaton accepts zero or more words formed by a sequence of the type '
para = 'The automaton accepts any number of words composed of sequences of the form '

  
paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

# linguaggio
#text, para, s1, s2, c, e1, e2, a += ''.join(linguaggio) + '. '
text, para, s1, s2, c, e1, e2, a = [v + ''.join(linguaggio) + '. ' for v in [text, para, s1, s2, c, e1, e2, a]]
wrong.append(text)

a += "I think that "
# stati
if (len(stati) > 1):
    #text, para, s1, s2, c, e1, e2, a += 'In total there are ' + str(len(stati)) + ' states: '
    text, para, s1, s2, c, e1, e2, a = [v + 'In total there are ' + str(len(stati)) + ' states: ' for v in [text, para, s1, s2, c, e1, e2, a]]
else:
    #text, para, s1, s2, c, e1, e2, a += 'In total there is only one state: '
    text, para, s1, s2, c, e1, e2, a = [v + 'In total there is only one state: ' for v in [text, para, s1, s2, c, e1, e2, a]]
for i in range(len(stati)):
    if (i == len(stati) - 1):
        #text, para, s1, s2, c, e1, e2, a += stati[i] + '. '
        text, para, s1, s2, c, e1, e2, a = [v + stati[i] + '. ' for v in [text, para, s1, s2, c, e1, e2, a]]
    elif (i == len(stati) - 2):
        #text, para, s1, s2, c, e2, a += stati[i] + ' and '
        text, para, s1, s2, c, e2, a = [v + stati[i] + ' and ' for v in [text, para, s1, s2, c, e2, a]]
        e1 += 'q10, ' + stati[i] + ' and '
    else:
        #text, para, s1, s2, c, e1, e2, a += stati[i] + ', '
        text, para, s1, s2, c, e1, e2, a = [v + stati[i] + ', ' for v in [text, para, s1, s2, c, e1, e2, a]]

a += "I think that "
# stato iniziale e stato finale
if (stato_iniziale != '' and stato_finale != ''):
    if (stato_iniziale == stato_finale):
        #text, s2, e1, e2, a += stato_iniziale + ' is both the initial and the final state.'
        text, s2, e1, e2, a = [v + stato_iniziale + ' is both the initial and the final state.' for v in [text, s2, e1, e2, a]]
        para += stato_iniziale + ' serves as both the initial and final state.'
        s1 += stati[2] + ' is both the initial and the final state.'
        c += stato_iniziale + ' is not both the initial and the final state.'
    else:
        #text, s2, e1, e2, a += 'The initial state is ' + stato_iniziale + ' and the final state is ' + stato_finale + '.'
        text, s2, e1, e2, a = [v + 'The initial state is ' + stato_iniziale + ' and the final state is ' + stato_finale + '.' for v in [text, s2, e1, e2, a]]
        para += 'The initial state is ' + stato_iniziale + ', while the final state is ' + stato_finale + '.'
        s1 += 'The initial state is ' + stato_finale + ' and the final state is ' + stato_iniziale + '.'
        c += 'The initial state is not ' + stato_iniziale + ' and the final state is not ' + stato_finale + '.'

wrong.append(text)

# transizioni
missing = missing_detail_trans(text, transizioni, transizioni_linguaggio)
a += " I think that"
#text, para, s1, s2, c, e1, e2, a += ' The transitions are: '
text, para, s1, s2, c, e1, e2, a = [v + ' The transitions are: ' for v in [text, para, s1, s2, c, e1, e2, a]]
for i in range(len(transizioni)):
    if (i == len(transizioni) - 1):
        #text, para, s1, c, e1, a += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
        text, para, s1, c, e1, a = [v + transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.' for v in [text, para, s1, c, e1, a]]
        e2 += 'q10 with value 10 goes to q11' + transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
        s2 += transizioni[i][1] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][0] + '.'
    else:
        #text, para, s1, s2, c, e1, e2, a += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + ', '
        text, para, s1, s2, c, e1, e2, a = [v + transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + ', ' for v in [text, para, s1, s2, c, e1, e2, a]]
t = add_typo(text)
text += svg

category['template'] = text
category['request'] = 'Can you describe me the automaton?'
category['wrong'] = wrong

paraphrase.append(para)
missing_detail.extend(missing)
swap.extend([s1, s2])
contradiction.append(c)
extra.extend([e1, e2])
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(t)
ambiguos.append(a)

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'automaton',
 'acts': {0: 'Ta:request'},
 'frame': '<frame></frame>',
 'template': 'The automaton accepts zero or more words formed by a sequence of the type 11000. In total there are 5 states: q0, q1, q2, q3 and q4. q0 is both the initial and the final state. The transitions are: q0 with value 1 goes to q1, q1 with value 1 goes to q2, q2 with value 0 goes to q3, q3 with value 0 goes to q4, q4 with value 0 goes to q0.<image>automi/automa.svg</image>',
 'request': 'Can you describe me the automaton?',
 'wrong': ['The automaton accepts zero or more words formed by a sequence of the type 11000. ',
  'The automaton accepts zero or more words formed by a sequence of the type 11000. In total there are 5 states: q0, q1, q2, q3 and q4. q0 is both the initial and the final state.'],
 'metamorph': {'paraphrase': ['The automaton accepts any number of words composed of sequences of the form 11000. In total there are 5 states: q0, q1, q2, q3 and q4. q0 serve

### Stati

In [63]:
# Quali sono gli stati
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": "<frame></frame>",
    "template": ""
}

paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

missing = missing_detail_states('', stati)

wrong = []
text = e = 'The automaton has ' + str(len(stati)) + ' states: '
para = 'The automaton consists of ' + str(len(stati)) + ' states: '
c = 'The automaton has not ' + str(len(stati)) + ' states: '
e += 'q10, '
a = 'I think the automaton has ' + str(len(stati)) + ' states, they should be: '

w_tmp = 'The automaton has ' + str(len(stati) - 2) + ' states: '
wrong.append(w_tmp)

for i in range(len(stati)):
    if (i == len(stati) - 1):
        #text, para, c, e += stati[i] + '.'
        text, para, c, e, a = [v + stati[i] + '.' for v in [text, para, c, e, a]]
    elif (i == len(stati) - 2):
        #text, para, c, e += stati[i] + ' and '
        text, para, c, e, a = [v + stati[i] + ' and ' for v in [text, para, c, e, a]]
    else:
        #text, para, c, e, w_tmp += stati[i] + ', '
        text, para, c, e, a, w_tmp = [v + stati[i] + ', ' for v in [text, para, c, e, a, w_tmp]]

t = add_typo(text)

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

text += svg
wrong.append(w_tmp)


category['template'] = text
category['request'] = 'How many states does the automaton have, and what are they?'
category['wrong'] = wrong

paraphrase.append(para)
missing_detail.extend(missing)
contradiction.append(c)
extra.append(e)
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(t)
ambiguos.append(a)

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '<frame></frame>',
 'template': 'The automaton has 5 states: q0, q1, q2, q3 and q4.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>',
 'request': 'How many states does the automaton have, and what are they?',
 'wrong': ['The automaton has 3 states: ',
  'The automaton has 3 states: q0, q1, q2, '],
 'metamorph': {'paraphrase': ['The automaton consists of 5 states: q0, q1, q2, q3 and q4.'],
  'missing_detail': ['The automaton has 1 states: q0.',
   'The automaton has 2 states: q0 and q1.',
   'The automaton has 3 states: q0, q1 and q2.',
 

### Transizioni

In [64]:
# quali sono le transizioni
category = {
    "intent": "fsa-practical",
    "argument": "transition",
    "acts": {
        0: "Ta:request",
    },
    "frame": """<frame></frame>""",
    "template": ""
}
wrong = []

paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1) #
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate) #
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot) #
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0) # 
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓) #
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0) #
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change) #
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta) #

text = s = e = 'The automaton has ' + str(len(transizioni)) + ' transitions: '
c = 'The automaton has not ' + str(len(transizioni)) + ' transitions: '
para = 'The automaton contains ' + str(len(transizioni)) + ' transitions: '
a = 'I think the automaton has ' + str(len(transizioni)) + ' transitions, they should be: '
missing = missing_detail_number_trans('', transizioni, transizioni_linguaggio)

w_tmp = 'The automaton has ' + str(len(transizioni) - 2) + ' transitions: '
wrong.append(w_tmp)

for i in range(len(transizioni)):
    if (i == len(transizioni) - 1):
        #text, para, c, a += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
        text, para, c, a = [v + transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.' for v in [text, para, c, a]]
        s += transizioni[i][1] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][0] + '.'
        e += 'q10 with value 10 goes to q11, ' + transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
    else:
        #text, para, s, c, e, a, w_tmp += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + ', '
        text, para, s, c, e, a, w_tmp = [v + transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + ', ' for v in [text, para, s, c, e, a, w_tmp]]


for transizione in transizioni:
    elemento = 'simbolo-' + transizione[0] + '-' + transizione[1]
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

t = add_typo(text)
text += svg
wrong.append(w_tmp)

category['template'] = text
category['request'] = 'How many transitions does the automaton have, and what are their values?'
category['wrong'] = wrong

paraphrase.append(para)
missing_detail.extend(missing)
swap.append(s)
contradiction.append(c)
extra.append(e)
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(t)
ambiguos.append(a)

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '<frame></frame>',
 'template': 'The automaton has 5 transitions: q0 with value 1 goes to q1, q1 with value 1 goes to q2, q2 with value 0 goes to q3, q3 with value 0 goes to q4, q4 with value 0 goes to q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3-q4</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4-q0</svgElement><image>automi/automa.svg</image>',
 'request': 'How many transitions does the automaton have, and what are their values?',
 'wrong': ['The automaton has 3 transitions: ',
  'The automaton has 3 transitions: q0 with value 1 goes to q1, q1 with value 1 goes to q2, q2 with value 0 goes to q3, q3 with val

### Stato iniziale

In [65]:
# qual è lo stato iniziale
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="?"/>
        </frame>
    """,
    "template": ""
}

paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)


text = 'The initial state is ' + stato_iniziale + '.'
elemento = 'simbolo-' + stato_iniziale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
category['request'] = 'Which is the initial state?'
categories.append(category)

paraphrase.append('The automaton starts in state ' + stato_iniziale + '.')
swap.append('The initial state is ' + stati[2] + '.')
contradiction.append('The initial state is not ' + stato_iniziale + '.')
extra.append('The initial state are ' + stato_iniziale + ' and in state q10.')
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(add_typo('The initial state is ' + stato_iniziale + '.'))
ambiguos.append('I think the initial state is ' + stato_iniziale + '.')

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="?"/>\n        </frame>\n    ',
 'template': 'The initial state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>',
 'request': 'Which is the initial state?',
 'metamorph': {'paraphrase': ['The automaton starts in state q0.'],
  'missing_detail': [],
  'swap': ['The initial state is q2.'],
  'contradiction': ['The initial state is not q0.'],
  'extra': ['The initial state are q0 and in state q10.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['The initial stat is q0.',
   'Theinitial state is q0.',
   'The jinitial state is q0.',
   'Thwe initial state is q0.'],
  'ambiguos': ['I think the initial state is q0.']}}

### Stato finale

In [66]:
# qual 'è lo stato finale?
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="finalStates" value="?"/>
        </frame>
    """,
    "template": ""
}
paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

text = 'The final state is ' + stato_finale + '.'
elemento = 'simbolo-' + stato_finale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
category['request'] = 'Which is the final state?'

paraphrase.append('The automaton ends in state ' + stato_finale + '.')
swap.append('The final state is ' + stati[2] + '.')
contradiction.append('The final state is not ' + stato_finale + '.')
extra.append('The final state are ' + stato_finale + ' and in state q10.')
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(add_typo('The final state is ' + stato_finale + '.'))
ambiguos.append('I think the final state is ' + stato_finale + '.')

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="finalStates" value="?"/>\n        </frame>\n    ',
 'template': 'The final state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>',
 'request': 'Which is the final state?',
 'metamorph': {'paraphrase': ['The automaton ends in state q0.'],
  'missing_detail': [],
  'swap': ['The final state is q2.'],
  'contradiction': ['The final state is not q0.'],
  'extra': ['The final state are q0 and in state q10.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['Th final state is q0.',
   'The final tate is q0.',
   'The final state is rq0.',
   'yThe final state is q0.'],
  'ambiguos': ['I think the final state is q0.']}}

### Stati e archi

In [67]:
# quanti stati iniziali e finali ci sono
category = {
    "intent": "fsa-practical",
    "argument": "automaton",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="?"/>
            <slot name="numberOfTransitions" value="?"/>
        </frame>
    """,
    "template": ""
}
paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

wrong = []
wrong.append('The automaton has ' + str(len(stati) + 2) + ' states and ' + str(len(transizioni) + 4) + ' transitions.')

text = 'The automaton has ' + str(len(stati)) + ' states and ' + str(len(transizioni)) + ' transitions.'
text += svg

category['template'] = text
category['request'] = 'How many states and transitions does the automaton have?'
category['wrong'] = wrong

paraphrase.append('The automaton consists of ' + str(len(stati)) + ' states and ' + str(len(transizioni)) + ' transitions.')
missing_detail.extend(['The automaton has ' + str(len(stati)) + ' states.', 'The automaton has ' + str(len(transizioni)) + ' transitions.'])
contradiction.append('The automaton has not ' + str(len(stati)) + ' states and ' + str(len(transizioni)) + ' transitions.')
extra.extend(['The automaton has ' + str(len(stati)+1) + ' states and ' + str(len(transizioni)) + ' transitions.', 'The automaton has ' + str(len(stati)) + ' states and ' + str(len(transizioni)+1) + ' transitions.'])
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(add_typo('The automaton has ' + str(len(stati)) + ' states and ' + str(len(transizioni)) + ' transitions.'))
ambiguos.append('I think the automaton has ' + str(len(stati)) + ' states and ' + str(len(transizioni)) + ' transitions.')

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'automaton',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="?"/>\n            <slot name="numberOfTransitions" value="?"/>\n        </frame>\n    ',
 'template': 'The automaton has 5 states and 5 transitions.<image>automi/automa.svg</image>',
 'request': 'How many states and transitions does the automaton have?',
 'wrong': ['The automaton has 7 states and 9 transitions.'],
 'metamorph': {'paraphrase': ['The automaton consists of 5 states and 5 transitions.'],
  'missing_detail': ['The automaton has 5 states.',
   'The automaton has 5 transitions.'],
  'swap': [],
  'contradiction': ['The automaton has not 5 states and 5 transitions.'],
  'extra': ['The automaton has 6 states and 5 transitions.',
   'The automaton has 5 states and 6 transitions.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['The automaton has 5 state and 5 transitions.',
   'The automatonhas 5 st

### Stato iniziale e stato finale ###

In [68]:
# qual è lo stato finale e quale lo stato iniziale
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="?""/>
            <slot name="finalStates" value="?"/>
        </frame>
    """,
    "template": ""
}

text = ''
paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

c = m1 = m2 = s = para = e = ''
# stato iniziale e stato finale
if (stato_iniziale != '' and stato_finale != ''):
    if (stato_iniziale == stato_finale):
        text += stato_iniziale + ' is the initial and the final state.'
        c = stato_iniziale + ' is not the initial and the final state.'
        m1 = stato_iniziale + ' is the initial state.'
        m2 = stato_iniziale + ' is the final state.'
        s = stati[2] + ' is the initial and the final state.'
        para = stato_iniziale + ' serves as both the initial and final state.'
    else:
        text += 'The initial state is ' + stato_iniziale + ' and the final state is ' + stato_finale + '.'
        s = 'The initial state is ' + stato_finale + ' and the final state is ' + stato_iniziale + '.'
        c = 'The initial state is not ' + stato_iniziale + ' and the final state is not ' + stato_finale + '.'
        m1 = stato_iniziale + ' is the initial state.'
        m2 = stato_finale + ' is the final state.'
        para = 'The automaton starts in state ' + stato_iniziale + ' and ends in state ' + stato_finale + '.'
    elemento = 'simbolo-' + stato_iniziale
    
    t = add_typo(text)
    e = text + ' The automata has two cycles.'
    a = 'I think that ' + text

    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

    if (not (stato_iniziale == stato_finale)):
        elemento2 = 'simbolo-' + stato_finale
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento2 + '</svgElement>'
else:
    text += "the initial state and the final state are not defined. "
    para = "The initial and final states are not defined."
    c = "the initial state and the final state are defined. "
    m1 = "the initial state is not defined. "
    m2 = "the final state is not defined. "
    t = add_typo(text)
    e = text + ' The automata has two cycles.'
    a = 'I think that ' + text
    
    


text += svg

category['template'] = text
category['request'] = 'What is the initial state and what is the final state?'

paraphrase.append(para)
missing_detail.extend([m1, m2])
swap.append(s)
contradiction.append(c)
extra.append(e)
off_topic.extend(['I like being a teacher', 'I like pizza'])
typo.extend(t)
ambiguos.append(a)

metamorph = {
    "paraphrase": paraphrase,
    "missing_detail": missing_detail,
    "swap": swap,
    "contradiction": contradiction,
    "extra": extra,
    "off_topic": off_topic,
    "typo": typo,
    "ambiguos": ambiguos
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="?""/>\n            <slot name="finalStates" value="?"/>\n        </frame>\n    ',
 'template': 'q0 is the initial and the final state.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>',
 'request': 'What is the initial state and what is the final state?',
 'metamorph': {'paraphrase': ['q0 serves as both the initial and final state.'],
  'missing_detail': ['q0 is the initial state.', 'q0 is the final state.'],
  'swap': ['q2 is the initial and the final state.'],
  'contradiction': ['q0 is not the initial and the final state.'],
  'extra': ['q0 is the initial and the final state. The automata has two cycles.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['q0 is the initial and the final tate.',
   'q0 is the initial and the finalstate.',
   'q0 is the 

### Transizione da questo a quello (tramite touch sugli stati)

In [ ]:
seen_pairs = set()  # tiene traccia delle coppie già elaborate

for stato in stati:
    for stato2 in stati:
        category = {
            "intent": "fsa-practical",
            "argument": "transition",
            #"acts": {0: "Ta:propositionalQuestion"},
            "acts": {0: "Ta:answer"},
            "frame": f"""
                <frame>
                    <slot name="states">
                        <slot-value value="{stato}"/>
                        <slot-value value="{stato2}"/>
                    </slot>
                </frame>
            """,
            "template": ""
        }
        wrong = []

        paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
        missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
        swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
        contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
        extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
        off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
        typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
        ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

        para = m = s = a = e = c = ''
        request = f'Is there a transition between {stato} and {stato2}, and what is its value?'

        # cerca le transizioni in entrambe le direzioni
        valore1 = next((value for s, value in transizioni_linguaggio if s == [stato, stato2]), None)
        valore2 = next((value for s, value in transizioni_linguaggio if s == [stato2, stato]), None)

        if valore1 and valore2:
            text = (f'There are transitions between {stato} and {stato2} in both directions: '
                    f'{stato}->{stato2} = {valore1}, {stato2}->{stato} = {valore2}.')
            para = (f'Transitions exist between {stato} and {stato2} in both directions: '
                    f'{stato}->{stato2} = {valore1}, and {stato2}->{stato} = {valore2}.')
            m = f'There are transitions between {stato} and {stato2} in both directions.'
            s = (f'There are transitions between {stato} and {stato2} in both directions: '
                    f'{stato2}->{stato} = {valore1}, {stato}->{stato2} = {valore2}.')
            c = (f'There are not transitions between {stato} and {stato2}: '
                    f'{stato}->{stato2} = {valore1}, {stato2}->{stato} = {valore2}.')
            e = (f'There are transitions between {stato} and {stato2} in both directions: '
                    f'{stato}->{stato2} = {valore1}, {stato2}->{stato} = {valore2}, q10->q11 = 10.')
            wrong.append(f'There is no transition between {stato} and {stato2}.')
            elemento = f'simbolo-{stato}-{stato2}'
            a = "I think that " + text
            t = add_typo(text)
            text += f'<svgElement style-name="stroke" style-value="#04ed00">{elemento}</svgElement>'
        elif valore1:
            text = f'The transition from {stato} to {stato2} has value {valore1}.'
            para = f'The transition from {stato} to {stato2} is labeled with {valore1}.'
            m = f'There is a transition from {stato} to {stato2}'
            s = f'The transition from {stato2} to {stato} has value {valore1}.'
            c = f'The transition from {stato} to {stato2} has not value {valore1}.'
            e = f'The transition from {stato} to {stato2} has value {valore1} and the transition from {stato2} to {stato} has value 5.'
            wrong.append(f'The transition from {stato} to {stato2} has value {valore1}0.')
            elemento = f'simbolo-{stato}-{stato2}'
            a = "I think that " + text
            t = add_typo(text)
            text += f'<svgElement style-name="stroke" style-value="#04ed00">{elemento}</svgElement>'
        elif valore2:
            text = f'The transition from {stato2} to {stato} has value {valore2}.'
            para = f'The transition from {stato2} to {stato} is labeled with {valore2}.'
            m = f'There is a transition from {stato2} to {stato}'
            s = f'The transition from {stato} to {stato2} has value {valore2}.'
            c = f'The transition from {stato2} to {stato} has not value {valore2}.'
            e = f'The transition from {stato2} to {stato} has value {valore2} and the transition from {stato} to {stato2} has value 5.'
            request = f'Is there a transition between {stato2} and {stato}, and what is its value?'
            wrong.append(f'The transition from {stato2} to {stato} has value {valore2}0.')
            elemento = f'simbolo-{stato2}-{stato}'
            a = "I think that " + text
            t = add_typo(text)
            text += f'<svgElement style-name="stroke" style-value="#04ed00">{elemento}</svgElement>'
        else:
            text = f'There is no transition between {stato} and {stato2}.'
            c = f'There is a transition between {stato} and {stato2}.'
            para = f'No transition exists between {stato} and {stato2}.'
            e = f'The transition from {stato2} to {stato} has value 5.'
            wrong.append(f'There are transitions between {stato} and {stato2} in both directions: {stato}->{stato2} = 5, {stato2}->{stato} = 8.')
            t = add_typo(text)
            a = "I think that " + text

        text += svg
        category['template'] = 'MUTATION ' + text
        category['request'] = request
        category['wrong'] = wrong

        paraphrase.append(para)
        missing_detail.append(m)
        swap.append(s)
        contradiction.append(c)
        extra.append(e)
        off_topic.extend(['I like being a teacher', 'I like pizza'])
        typo.extend(t)
        ambiguos.append(a)

        metamorph = {
            "paraphrase": paraphrase,
            "missing_detail": missing_detail,
            "swap": swap,
            "contradiction": contradiction,
            "extra": extra,
            "off_topic": off_topic,
            "typo": typo,
            "ambiguos": ambiguos
        }
        category['metamorph'] = metamorph
        
        print(category)
        categories.append(category)

{'intent': 'fsa-practical', 'argument': 'transition', 'acts': {0: 'Ta:propositionalQuestion'}, 'frame': '<frame></frame>', 'template': 'MUTATION There is no transition between q0 and q0.<image>automi/automa.svg</image>', 'request': 'Is there a transition between q0 and q0, and what is its value?', 'wrong': ['There are transitions between q0 and q0 in both directions: q0->q0 = 5, q0->q0 = 8.'], 'metamorph': {'paraphrase': ['No transition exists between q0 and q0.'], 'missing_detail': [''], 'swap': [''], 'contradiction': ['There is a transition between q0 and q0.'], 'extra': ['The transition from q0 to q0 has value 5.'], 'off_topic': ['I like being a teacher', 'I like pizza'], 'typo': ['There is no ransition between q0 and q0.', 'There is no transtion between q0 and q0.', 'There is no transition between q0 and tq0.', 'There is no ztransition between q0 and q0.'], 'ambiguos': ['I think that There is no transition between q0 and q0.']}}
{'intent': 'fsa-practical', 'argument': 'transition',

In [70]:
# transizione tra stato X e stato Y
category = {
    "intent": "fsa-practical",
    "argument": "transition",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="states">
                <slot-value value="*"/>
                <slot-value value="*"/>
            </slot>
        </frame>
    """,
    "template": ""
}

text = 'the transition is not defined.'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="states">\n                <slot-value value="*"/>\n                <slot-value value="*"/>\n            </slot>\n        </frame>\n    ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

In [71]:
for stato in stati:
    # transizione tra stato X e stato Y
    category = {
        "intent": "fsa-practical",
        "argument": "transition",
        "acts": {
            0: "Ta:request",
        },
        "frame": f"""
            <frame>
                <slot name="states">
                    <slot-value value="{stato}"/>
                    <slot-value value="*"/>
                </slot>
            </frame>
        """,
        "template": ""
    }

    text = 'the transition is not defined.'
    text += svg

    category['template'] = text
    print(category)
    categories.append(category)

{'intent': 'fsa-practical', 'argument': 'transition', 'acts': {0: 'Ta:request'}, 'frame': '\n            <frame>\n                <slot name="states">\n                    <slot-value value="q0"/>\n                    <slot-value value="*"/>\n                </slot>\n            </frame>\n        ', 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}
{'intent': 'fsa-practical', 'argument': 'transition', 'acts': {0: 'Ta:request'}, 'frame': '\n            <frame>\n                <slot name="states">\n                    <slot-value value="q1"/>\n                    <slot-value value="*"/>\n                </slot>\n            </frame>\n        ', 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}
{'intent': 'fsa-practical', 'argument': 'transition', 'acts': {0: 'Ta:request'}, 'frame': '\n            <frame>\n                <slot name="states">\n                    <slot-value value="q2"/>\n                    <slot-value value="*"/>\

### Da ... a ...

In [72]:
for stato in stati:
    for stato2 in stati:
        category = {
            "intent": "fsa-practical",
            "argument": "transition",
            "acts": {
                0: "Ta:propositionalQuestion",
            },
            "frame": f"""
                <frame>
                    <slot name="transitions">
                        <slot-values>
                            <slot-value value="{stato}"/>
                            <slot-value value="{stato2}"/>
                            <slot-value value="?"/>
                        </slot-values>
                    </slot>
                </frame>
            """,
            "template": ""
        }

        wrong = []

        paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
        missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
        swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
        contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
        extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
        off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
        typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
        ambiguos = []  # Ambiguous / vague (accuracy media, relevancy alta)

        para = s = c = e = a = ''

        flag = False
        valore = ''
        for sw, value in transizioni_linguaggio:
            if (sw == [stato, stato2]):
                flag = True
                valore = value

        if (flag):
            wrong.append('there is no transition between ' + stato + ' and ' + stato2 + '.')
            text = 'The transition between ' + stato + ' and ' + stato2 + ' is with value ' + valore + '.'
            para = 'The transition between ' + stato + ' and ' + stato2 + ' has a value of ' + valore + '.'
            if stato != stato2:
                s = 'The transition between ' + stato2 + ' and ' + stato + ' is with value ' + valore + '.'
            e = 'The transition between ' + stato + ' and ' + stato2 + ' is with value ' + valore + ' and the transition between ' + stato2 + ' and ' + stato + ' is with value 5.'
            c = 'The transition between ' + stato + ' and ' + stato2 + ' is not with value ' + valore + '.'
            t = add_typo(text)
            elemento = 'simbolo-' + stato + '-' + stato2
            a = 'I think the transition between ' + stato + ' and ' + stato2 + ' should be with value ' + valore + '.'
            text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
        else:
            text = 'There is no transition between ' + stato + ' and ' + stato2 + '.'
            para = 'There is no transition from ' + stato + ' to ' + stato2 + '.'
            if stato != stato2:
                s = 'There is no transition between ' + stato2 + ' and ' + stato + '.'
            c = 'There is a transition between ' + stato + ' and ' + stato2 + '.'
            e = 'There is no transition between ' + stato + ' and ' + stato2 + ' but the automata has 15 transitions in total.'
            t = add_typo(text)
            a = 'I think that there is no transition between ' + stato + ' and ' + stato2 + '.'
            wrong.append('The transition between ' + stato + ' and ' + stato2 + ' is with value 5.')

        text += svg

        category['template'] = text
        category['request'] = f'What is the value of the between {stato} and {stato2}?'
        category['wrong'] = wrong


        paraphrase.append(para)
        swap.append(s)
        contradiction.append(c)
        extra.append(e)
        off_topic.extend(['I like being a teacher', 'I like pizza'])
        typo.extend(t)
        ambiguos.append(a)

        metamorph = {
            "paraphrase": paraphrase,
            "missing_detail": missing_detail,
            "swap": swap,
            "contradiction": contradiction,
            "extra": extra,
            "off_topic": off_topic,
            "typo": typo,
            "ambiguos": ambiguos
        }
        category['metamorph'] = metamorph

        categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n                <frame>\n                    <slot name="transitions">\n                        <slot-values>\n                            <slot-value value="q4"/>\n                            <slot-value value="q4"/>\n                            <slot-value value="?"/>\n                        </slot-values>\n                    </slot>\n                </frame>\n            ',
 'template': 'There is no transition between q4 and q4.<image>automi/automa.svg</image>',
 'request': 'What is the value of the between q4 and q4?',
 'wrong': ['The transition between q4 and q4 is with value 5.'],
 'metamorph': {'paraphrase': ['There is no transition from q4 to q4.'],
  'missing_detail': [],
  'swap': [''],
  'contradiction': ['There is a transition between q4 and q4.'],
  'extra': ['There is no transition between q4 and q4 but the automata has 15 transitions in total.'],
  'off_topic':

### No transazione in quanto non esistono entrambi gli stati

In [73]:
# transizione tra stato X e stato Y
category = {
    "intent": "fsa-practical",
    "argument": "transition",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="transitions">
                <slot-values>
                    <slot-value value="*"/>
                    <slot-value value="*"/>
                    <slot-value value="?"/>
                </slot-values>
            </slot>
        </frame>
    """,
    "template": ""
}

text = 'the transition is not defined.'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="transitions">\n                <slot-values>\n                    <slot-value value="*"/>\n                    <slot-value value="*"/>\n                    <slot-value value="?"/>\n                </slot-values>\n            </slot>\n        </frame>\n    ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

### No transazione sicura in quanto non esiste lo stato di arrivo

In [74]:
for stato in stati:
    # transizione tra stato X e stato Y
    category = {
        "intent": "fsa-practical",
        "argument": "transition",
        "acts": {
            0: "Ta:request",
        },
        "frame": f"""
            <frame>
                <slot name="transitions">
                    <slot-values>
                        <slot-value value="{stato}"/>
                        <slot-value value="*"/>
                        <slot-value value="?"/>
                    </slot-values>
                </slot>
            </frame>
        """,
        "template": ""
    }

    text = 'the transition is not defined.'
    text += svg

    category['template'] = text
    categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n            <frame>\n                <slot name="transitions">\n                    <slot-values>\n                        <slot-value value="q4"/>\n                        <slot-value value="*"/>\n                        <slot-value value="?"/>\n                    </slot-values>\n                </slot>\n            </frame>\n        ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

### No transazione sicura in quanto non esiste lo stato di partenza

In [75]:
for stato in stati:
    # transizione tra stato X e stato Y
    category = {
        "intent": "fsa-practical",
        "argument": "transition",
        "acts": {
            0: "Ta:request",
        },
        "frame": f"""
            <frame>
                <slot name="transitions">
                    <slot-values>
                        <slot-value value="*"/>
                        <slot-value value="{stato}"/>
                        <slot-value value="?"/>
                    </slot-values>
                </slot>
            </frame>
        """,
        "template": ""
    }

    text = 'the transition is not defined.'
    text += svg

    category['template'] = text
    categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n            <frame>\n                <slot name="transitions">\n                    <slot-values>\n                        <slot-value value="*"/>\n                        <slot-value value="q4"/>\n                        <slot-value value="?"/>\n                    </slot-values>\n                </slot>\n            </frame>\n        ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

### ESISTE...

#### Stati esistenti

In [76]:
for stato in stati:
    category = {
        "intent": "fsa-practical",
        "argument": "transition",
        "acts": {
            0: "Ta:propositionalQuestion",
        },
        "frame": f"""
            <frame>
                <slot name="states">
                    <slot-value value="{stato}"/>
                </slot>
            </frame>
        """,
        "template": ""
    }

    paraphrase = []  # Paraphrase (accuracy≈1, relevancy≈1, correctness≈1)
    missing_detail = []  # Missing details (partial) (accuracy ↓ moderate, relevancy ≈1, correctness ↓ moderate)
    swap = []  # Wrong fact (single swap) (accuracy ↓ a lot, relevancy ≈1, correctness ↓ a lot)
    contradiction = []  # Contradiction (accuracy ≈0, relevancy ≈1, correctness ≈0)
    extra = []  # extra invented facts (accuracy ↓, could be high or not, correctness ↓)
    off_topic = []  # Off-topic / Irrelevant (accuracy ≈0, relevancy ≈0, correctness ≈0)
    typo = []  # Noisy / token-level changes (accuracy small change, correctness small change)
    ambiguos = []  # Ambiguous / vague (accuracy medium, relevancy high)

    text = f'The state {stato} exists.' 
    t = add_typo(text)
    a = 'I think that ' + text
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
    text += svg

    category['template'] = text

    # added
    category['request'] = f'Does state {stato} exist?'
    category['wrong'] = [f'The state {stato} does not exist.']

    paraphrase.append(f'The state {stato} is defined.')
    contradiction.append(f'The state {stato} is not defined.')
    extra.append(f'The state {stato} exists and is connected to q10.')
    off_topic.extend(['I like being a teacher', 'I like pizza'])
    typo.extend(t)
    ambiguos.append(a)

    metamorph = {
        "paraphrase": paraphrase,
        "missing_detail": missing_detail,
        "swap": swap,
        "contradiction": contradiction,
        "extra": extra,
        "off_topic": off_topic,
        "typo": typo,
        "ambiguos": ambiguos
    }
    category['metamorph'] = metamorph

    categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n            <frame>\n                <slot name="states">\n                    <slot-value value="q4"/>\n                </slot>\n            </frame>\n        ',
 'template': 'The state q4 exists.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>',
 'request': 'Does state q4 exist?',
 'wrong': ['The state q4 does not exist.'],
 'metamorph': {'paraphrase': ['The state q4 is defined.'],
  'missing_detail': [],
  'swap': [],
  'contradiction': ['The state q4 is not defined.'],
  'extra': ['The state q4 exists and is connected to q10.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['The stae q4 exists.',
   'Thestate q4 exists.',
   'The state q4 exisots.',
   'Tehe state q4 exists.'],
  'ambiguos': ['I think that The state q4 exists.']}}

#### Stato iniziale esiste

In [77]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="{stato_iniziale}"/>
        </frame>
    """,
    "template": ""
}

text = 'The initial state is ' + stato_iniziale + '.'
elemento = 'simbolo-' + stato_iniziale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
category['request'] = f'Which is the initial state?'
category['wrong'] = [f'The initial state does not exist.']

metamorph = {
    "paraphrase": ['The starting state is ' + stato_iniziale + '.'],
    "missing_detail": [],
    "swap": [],
    "contradiction": ['The initial state is not ' + stato_iniziale + '.'],
    "extra": ['The initial state is ' + stato_iniziale + ' and the final state does not exist.'],
    "off_topic": ['I like being a teacher', 'I like pizza'],
    "typo": add_typo('The initial state is ' + stato_iniziale + '.'),
    "ambiguos": ['I think that the initial state is ' + stato_iniziale + '.']
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="q0"/>\n        </frame>\n    ',
 'template': 'The initial state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>',
 'request': 'Which is the initial state?',
 'wrong': ['The initial state does not exist.'],
 'metamorph': {'paraphrase': ['The starting state is q0.'],
  'missing_detail': [],
  'swap': [],
  'contradiction': ['The initial state is not q0.'],
  'extra': ['The initial state is q0 and the final state does not exist.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['The intial state is q0.',
   'The initil state is q0.',
   'The initial state iis q0.',
   'The initial state isz q0.'],
  'ambiguos': ['I think that the initial state is q0.']}}

#### Stato iniziale non esiste

In [78]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="*"/>
        </frame>
    """,
    "template": ""
}

text = 'It is not the initial state.'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="*"/>\n        </frame>\n    ',
 'template': 'It is not the initial state.<image>automi/automa.svg</image>'}

#### Stato finale

In [79]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="finalStates">
                <slot-value value="{stato_finale}"/>
            </slot>
        </frame>
    """,
    "template": ""
}



text = 'The final state is ' + stato_finale + '.'
elemento = 'simbolo-' + stato_finale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
category['request'] = f'Which is the final state?'
category['wrong'] = [f'The final state does not exist.']

metamorph = {
    "paraphrase": ['The ending state is ' + stato_finale + '.'],
    "missing_detail": [],
    "swap": [],
    "contradiction": ['The final state is not ' + stato_finale + '.'],
    "extra": ['The final state is ' + stato_finale + ' and the initial state does not exist.'],
    "off_topic": ['I like being a teacher', 'I like pizza'],
    "typo": add_typo('The final state is ' + stato_finale + '.'),
    "ambiguos": ['I think that the final state is ' + stato_finale + '.']
}
category['metamorph'] = metamorph

categories.append(category)
category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="finalStates">\n                <slot-value value="q0"/>\n            </slot>\n        </frame>\n    ',
 'template': 'The final state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>',
 'request': 'Which is the final state?',
 'wrong': ['The final state does not exist.'],
 'metamorph': {'paraphrase': ['The ending state is q0.'],
  'missing_detail': [],
  'swap': [],
  'contradiction': ['The final state is not q0.'],
  'extra': ['The final state is q0 and the initial state does not exist.'],
  'off_topic': ['I like being a teacher', 'I like pizza'],
  'typo': ['The final state is q0',
   'The final tate is q0.',
   'The final sbtate is q0.',
   'The zfinal state is q0.'],
  'ambiguos': ['I think that the final state is q0.']}}

#### Non esistente

In [80]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="finalStates">
                <slot-value value="*"/>
            </slot>
        </frame>
    """,
    "template": ""
}



text = 'The state is not the final state.'
text += svg

category['template'] = text
categories.append(category)
category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="finalStates">\n                <slot-value value="*"/>\n            </slot>\n        </frame>\n    ',
 'template': 'The state is not the final state.<image>automi/automa.svg</image>'}

### Quanti stati

In [81]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="?"/>
        </frame>
    """,
    "template": ""
}

text = 'This automaton has ' + str(len(stati)) + ' states.'

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
category['request'] = f'How many states does this automaton have?'
category['wrong'] = ['This automaton has ' + str(len(stati) + 1) + ' states']

metamorph = {
    "paraphrase": ['This automaton consists of ' + str(len(stati)) + ' states.'],
    "missing_detail": [],
    "swap": [],
    "contradiction": ['This automaton has not ' + str(len(stati)) + ' states.'],
    "extra": ['This automaton has ' + str(len(stati)) + ' states and 10 transitions.'],
    "off_topic": ['I like being a teacher', 'I like pizza'],
    "typo": add_typo('This automaton has ' + str(len(stati)) + ' states.'),
    "ambiguos": ['I think that this automaton has ' + str(len(stati)) + ' states.']
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="?"/>\n        </frame>\n    ',
 'template': 'This automaton has 5 states.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>',
 'request': 'How many states does this automaton have?',
 'wrong': ['This automaton has 6 states'],
 'metamorph': {'paraphrase': ['This automaton consists of 5 states.'],
  'missing_detail': [],
  'swap': [],
  'contradiction': ['This automaton has not 5 states.'],
  'extra': ['This automaton has 5 states and 10 transitions.'],
  'off_topic': ['I lik

### Gli stati sono ... (risposta giusta)

In [82]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="{str(len(stati))}"/>
        </frame>
    """,
    "template": ""
}

text = f'There are {str(len(stati))} states in this automaton.'

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
category['request'] = f'How many states does this automaton have?'
category['wrong'] = ['This automaton has ' + str(len(stati) + 1) + ' states']

metamorph = {
    "paraphrase": [f'This automaton has {str(len(stati))} states.'],
    "missing_detail": [],
    "swap": [],
    "contradiction": [f'There are not {str(len(stati))} states in this automaton.'],
    "extra": [f'There are {str(len(stati))} states in this automaton and 10 transitions.'],
    "off_topic": ['I like being a teacher', 'I like pizza'],
    "typo": add_typo(f'There are {str(len(stati))} states in this automaton.'),
    "ambiguos": [f'I think that there are {str(len(stati))} states in this automaton.']
}
category['metamorph'] = metamorph

categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="5"/>\n        </frame>\n    ',
 'template': 'There are 5 states in this automaton.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>',
 'request': 'How many states does this automaton have?',
 'wrong': ['This automaton has 6 states'],
 'metamorph': {'paraphrase': ['This automaton has 5 states.'],
  'missing_detail': [],
  'swap': [],
  'contradiction': ['There are not 5 states in this automaton.'],
  'extra': ['There are 5 states in this automaton and 10 trans

### Gli stati sono ... (risposta sbagliata)

In [83]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="*"/>
        </frame>
    """,
    "template": ""
}

text = 'The number of states is incorrect.'

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="*"/>\n        </frame>\n    ',
 'template': 'The number of states is incorrect.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>'}

## Generazione AIML

In [84]:
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom
import json

def build_aiml(categories, out_path):
    root = ET.Element('aiml')

    for cat in categories:
        c = ET.SubElement(root, 'category')
        if 'intent' in cat:
            c.set('intent', cat['intent'])
        if 'argument' in cat:
            c.set('argument', cat['argument'])
        if 'wrong' in cat:
            c.set('wrong', cat['wrong'])
        if 'request' in cat:
            c.set('request', cat['request'])
        
        if 'metamorph' in cat:
            metamorph_str = json.dumps(cat['metamorph'], ensure_ascii=False)
            c.set('metamorph', metamorph_str)

        # acts
        acts_el = ET.SubElement(c, 'acts')
        for time, act in cat.get('acts', {}).items():
            act_el = ET.SubElement(acts_el, 'act', {'time': str(time)})
            act_el.text = act

        # frame: proviamo a parsare come XML, altrimenti lo inseriamo come testo dentro <frame>
        frame_str = cat.get('frame', '').strip()
        if frame_str:
            try:
                frame_elem = ET.fromstring(frame_str)
            except ET.ParseError:
                # fallback: crea frame e metti dentro il testo raw
                f_el = ET.SubElement(c, 'frame')
                f_el.text = frame_str
            else:
                # append dell'elemento frame già parsato (mantiene eventuali slot/attributi)
                c.append(frame_elem)
        else:
            # assicuriamoci che esista comunque <frame></frame>
            ET.SubElement(c, 'frame')

        # template: se il testo della template è un frammento XML valido, lo parsiamo e trasferiamo i figli
        template_el = ET.SubElement(c, 'template')
        tpl = cat.get('template', '')

        if tpl.strip():
            # wrapper per poter parsare testo misto + tag
            try:
                fragment = ET.fromstring(f'<fragment>{tpl}</fragment>')
            except ET.ParseError:
                # fallback: testo semplice (verrà escape-ato correttamente)
                template_el.text = tpl
            else:
                # testo prima del primo figlio
                template_el.text = fragment.text
                # append di ogni figlio (con le eventuali .tail preservate)
                for child in list(fragment):
                    template_el.append(child)
        # altrimenti template rimane vuoto

    # Serializziamo con minidom per ottenere pretty print
    rough = ET.tostring(root, encoding='utf-8')
    dom = minidom.parseString(rough)

    # Forza <frame></frame> (senza self-closing) aggiungendo un textnode vuoto quando il frame è veramente vuoto
    for frame_node in dom.getElementsByTagName('frame'):
        if not frame_node.hasChildNodes():
            frame_node.appendChild(dom.createTextNode(''))

    pretty = dom.toprettyxml(indent='    ', encoding='utf-8')
    with open(out_path, 'wb') as f:
        f.write(pretty)

# Esegui la generazione
file_path = 'metamorphed/automa.aiml'
build_aiml(categories, file_path)
print(f"✅ File AIML+ generato in: {file_path}")

✅ File AIML+ generato in: metamorphed/automa.aiml
